# KMM-Scandaten Exploration & Preprocessing

Exploration realer Scandaten vom Koordinatenmessgerät (Hexagon / PC-DMIS).  
Dateiformat: `.xyz` mit erweitertem Header (Scanlinien-Normalen).

**Ziele:**
1. Laden und Visualisieren der rohen Punktwolke
2. Analyse der Scanlinien-Metadaten (Richtung, Dichte)
3. Preprocessing-Steps einzeln anwenden und bewerten
4. Geometrische Plausibilitätsprüfung (RANSAC)
5. Vergleich Roh vs. Preprocessed

**Verwendung:** `SCAN_FILE` in Zelle 2 anpassen, dann alle Zellen ausführen.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd

project_root = Path().resolve().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from schweiss_ki.preprocessing import (
    PreprocessingPipeline,
    StatisticalOutlierFilter,
    RadiusOutlierFilter,
    VoxelGridDownsampler,
    NormalEstimator,
)
from schweiss_ki.core.data_structures import WeldVolumeModel

print('Imports OK')

## 1 – Konfiguration

In [ ]:
SCAN_FILE      = Path('../data/raw/cmm_scans/Schweißspalt_1,0_auf_2,5.xyz')
VIZ_MAX_POINTS = 50_000

print(f'Scan-Datei: {SCAN_FILE}')
print(f'Existiert:  {SCAN_FILE.exists()}')

## 2 – XYZ Laden & Header parsen

Das PC-DMIS `.xyz`-Format hat Scanlinien-Header im Format:
```
L{nr}##{color1}##{color2}##{nx}##{ny}##{nz}
```
Die Datenpunkte haben 7 Spalten: `x, y, z, vx, vy, vz, col7` (Bedeutung col7 tbd).

Open3D liest die ersten 3 Spalten (XYZ) direkt – die Header werden übersprungen.

In [ ]:
# --- Open3D: direktes Laden (nur XYZ) ---
pcd_raw = o3d.io.read_point_cloud(str(SCAN_FILE))
pts_raw = np.asarray(pcd_raw.points)

bbox = pcd_raw.get_axis_aligned_bounding_box()
extent = bbox.get_extent()

print(f'Punkte:       {len(pcd_raw.points):,}')
print(f'Bounding Box: {extent[0]:.1f} x {extent[1]:.1f} x {extent[2]:.1f} mm')
print(f'Hat Normalen: {pcd_raw.has_normals()}')
print(f'Hat Farben:   {pcd_raw.has_colors()}')

In [ ]:
# --- Erweiterter Parser: Header-Metadaten + alle 7 Spalten ---
def parse_cmm_xyz(filepath: Path) -> dict:
    """
    Parst PC-DMIS .xyz mit Scanlinien-Headern.
    
    Returns:
        dict mit:
        - 'points': np.ndarray (N, 3) – XYZ Koordinaten
        - 'all_columns': np.ndarray (N, 7) – alle Spalten
        - 'scan_line_id': np.ndarray (N,) – Scanlinien-Zuordnung pro Punkt
        - 'scan_normals': np.ndarray (N, 3) – Scannormale pro Punkt (vom Header)
        - 'headers': list[dict] – geparste Header-Infos
    """
    points = []
    all_columns = []
    scan_line_ids = []
    scan_normals = []
    headers = []
    
    current_line_id = None
    current_normal = None
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            if line.startswith('L'):
                parts = line.split('##')
                current_line_id = parts[0]
                current_normal = [float(parts[3]), float(parts[4]), float(parts[5])]
                headers.append({
                    'line_id': current_line_id,
                    'color1': int(parts[1]),
                    'color2': int(parts[2]),
                    'normal': current_normal,
                })
            else:
                vals = line.split()
                if len(vals) >= 3:
                    row = [float(v) for v in vals]
                    points.append(row[:3])
                    all_columns.append(row)
                    scan_line_ids.append(current_line_id)
                    scan_normals.append(current_normal if current_normal else [0,0,0])
    
    return {
        'points': np.array(points),
        'all_columns': np.array(all_columns),
        'scan_line_id': np.array(scan_line_ids),
        'scan_normals': np.array(scan_normals),
        'headers': headers,
    }

parsed = parse_cmm_xyz(SCAN_FILE)

print(f'Punkte:         {parsed["points"].shape}')
print(f'Alle Spalten:   {parsed["all_columns"].shape}')
print(f'Scanlinien:     {len(parsed["headers"]):,}')

# Unique Scan-Richtungen
unique_normals = np.unique(parsed['scan_normals'], axis=0)
print(f'\nScan-Richtungen ({len(unique_normals)}):')
for n in unique_normals:
    mask = np.all(np.isclose(parsed['scan_normals'], n), axis=1)
    print(f'  n=[{n[0]:+.4f}, {n[1]:+.4f}, {n[2]:+.4f}]  →  {mask.sum():,} Punkte')

## 3 – Hilfsfunktionen

In [ ]:
def subsample(pcd, max_points=VIZ_MAX_POINTS):
    """Subsampled Punktwolke für Plotly-Visualisierung."""
    pts = np.asarray(pcd.points)
    if len(pts) > max_points:
        idx = np.random.default_rng(42).choice(len(pts), max_points, replace=False)
        pts = pts[idx]
    return pts


def plot_single(pcd, title, color='steelblue'):
    """3D-Scatter einer einzelnen Punktwolke."""
    pts = subsample(pcd)
    fig = go.Figure(go.Scatter3d(
        x=pts[:,0], y=pts[:,1], z=pts[:,2],
        mode='markers',
        marker=dict(size=1.5, color=color, opacity=0.7),
        name=f'{len(pcd.points):,} Punkte',
    ))
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
            aspectmode='data',
        ),
        height=600,
    )
    fig.show()


def plot_before_after(pcd_before, pcd_after, title,
                      color_before='steelblue', color_after='tomato'):
    """3D-Scatter Vergleich vorher/nachher."""
    pts_b = subsample(pcd_before)
    pts_a = subsample(pcd_after)
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=pts_b[:,0], y=pts_b[:,1], z=pts_b[:,2],
        mode='markers', marker=dict(size=1, color=color_before, opacity=0.3),
        name=f'Vorher: {len(pcd_before.points):,}',
    ))
    fig.add_trace(go.Scatter3d(
        x=pts_a[:,0], y=pts_a[:,1], z=pts_a[:,2],
        mode='markers', marker=dict(size=1.5, color=color_after, opacity=0.7),
        name=f'Nachher: {len(pcd_after.points):,}',
    ))
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
            aspectmode='data',
        ),
        height=600,
    )
    fig.show()

## 4 – Rohe Punktwolke

In [ ]:
plot_single(pcd_raw, f'Roh: {SCAN_FILE.stem} ({len(pcd_raw.points):,} Punkte)')

### 4.1 – Scan-Richtungen visualisieren

Die KMM hat aus 2 Richtungen (~±30° zur Vertikalen) gescannt, um die V-Naht abzubilden.

In [ ]:
# Punkte nach Scanrichtung einfärben
normals = parsed['scan_normals']
direction_label = np.where(normals[:, 0] < 0, 'Scan A (n_x < 0)', 'Scan B (n_x > 0)')

pts = parsed['points']
fig = go.Figure()
for label, color in [('Scan A (n_x < 0)', '#2196F3'), ('Scan B (n_x > 0)', '#FF5722')]:
    mask = direction_label == label
    fig.add_trace(go.Scatter3d(
        x=pts[mask, 0], y=pts[mask, 1], z=pts[mask, 2],
        mode='markers',
        marker=dict(size=1.2, color=color, opacity=0.6),
        name=f'{label}: {mask.sum():,} Punkte',
    ))
fig.update_layout(
    title='Scan-Richtungen (2 Winkel, ~±30° zur Vertikalen)',
    scene=dict(
        xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
        aspectmode='data',
    ),
    height=600,
)
fig.show()

### 4.2 – Querschnitte (Y-Slices)

In [ ]:
y_min, y_max = pts[:, 1].min(), pts[:, 1].max()
y_positions = np.linspace(y_min + 5, y_max - 5, 6)
SLICE_THICKNESS = 0.5  # mm (±)

# Globale Achsenbereiche berechnen
x_global = [pts[:, 0].min() - 1, pts[:, 0].max() + 1]
z_global = [pts[:, 2].min() - 0.5, pts[:, 2].max() + 0.5]

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'Y ≈ {y:.0f} mm' for y in y_positions],
)

for i, y in enumerate(y_positions):
    row = i // 3 + 1
    col = i % 3 + 1
    mask = np.abs(pts[:, 1] - y) < SLICE_THICKNESS
    sl = pts[mask]
    fig.add_trace(
        go.Scatter(
            x=sl[:, 0], y=sl[:, 2],
            mode='markers',
            marker=dict(size=2, color='steelblue'),
            name=f'{mask.sum()} pts',
            showlegend=False,
        ),
        row=row, col=col,
    )
    fig.update_xaxes(title_text='X (mm)', range=x_global, row=row, col=col)
    fig.update_yaxes(title_text='Z (mm)', range=z_global, row=row, col=col)

fig.update_layout(
    title=f'Querschnitte (XZ) bei verschiedenen Y-Positionen (±{SLICE_THICKNESS} mm)',
    height=700,
)
fig.show()

### 4.3 – Höhenprofil (Z-Verteilung)

In [ ]:
fig = px.histogram(
    x=pts[:, 2], nbins=150,
    title='Z-Wert-Verteilung (Höhenprofil)',
    labels={'x': 'Z (mm)', 'y': 'Anzahl'},
    color_discrete_sequence=['steelblue'],
)
fig.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='Z=0')
fig.update_layout(height=400)
fig.show()

### 4.4 – Punktdichte pro Scanlinie

In [ ]:
line_ids, counts = np.unique(parsed['scan_line_id'], return_counts=True)

fig = px.histogram(
    x=counts, nbins=50,
    title=f'Punkte pro Scanlinie ({len(line_ids):,} Linien)',
    labels={'x': 'Punkte pro Linie', 'y': 'Anzahl Linien'},
    color_discrete_sequence=['steelblue'],
)
fig.update_layout(height=400)
fig.show()

print(f'Min: {counts.min()}, Max: {counts.max()}, Median: {np.median(counts):.0f}, Mean: {counts.mean():.1f}')

---
## 5 – Preprocessing Step-für-Step

Verwendet die bestehenden Preprocessing-Klassen aus `schweiss_ki.preprocessing`.  
Parameter hier zunächst mit Default-Werten — Tuning nach visueller Inspektion.

### 5.1 – Statistical Outlier Filter

Für jeden Punkt wird der mittlere Abstand zu seinen `nb_neighbors` nächsten Nachbarn berechnet.
Über alle Punkte ergibt sich ein globaler Mittelwert μ und eine Standardabweichung σ dieser Abstände.
Punkte, deren mittlerer Nachbar-Abstand über **μ + std_ratio × σ** liegt, werden als Outlier entfernt.

→ Erwischt Punkte, die **räumlich isolierter** sind als der Rest der Wolke.

In [ ]:
NB_NEIGHBORS = 20
STD_RATIO    = 2.0

# --- Filter anwenden (manuell für Outlier-Indizes) ---
pcd_stat, inlier_idx = pcd_raw.remove_statistical_outlier(
    nb_neighbors=NB_NEIGHBORS, std_ratio=STD_RATIO
)

all_idx = set(range(len(pcd_raw.points)))
inlier_set = set(inlier_idx)
outlier_idx = sorted(all_idx - inlier_set)

pts_all = np.asarray(pcd_raw.points)
pts_inlier = pts_all[list(inlier_set)]
pts_outlier = pts_all[outlier_idx]

print(f'Vorher:   {len(pts_all):,} Punkte')
print(f'Nachher:  {len(pts_inlier):,} Punkte')
print(f'Entfernt: {len(pts_outlier):,} ({len(pts_outlier)/len(pts_all)*100:.1f}%)')

In [ ]:
# --- 3D-Plot: Inlier grau, Outlier rot hervorgehoben ---
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=pts_inlier[:, 0], y=pts_inlier[:, 1], z=pts_inlier[:, 2],
    mode='markers',
    marker=dict(size=1, color='#B0BEC5', opacity=0.3),
    name=f'Beibehalten ({len(pts_inlier):,})',
))
fig.add_trace(go.Scatter3d(
    x=pts_outlier[:, 0], y=pts_outlier[:, 1], z=pts_outlier[:, 2],
    mode='markers',
    marker=dict(size=4, color='#E53935', opacity=0.9),
    name=f'Entfernt ({len(pts_outlier):,})',
))
fig.update_layout(
    title=f'Statistical Outlier Filter (nb={NB_NEIGHBORS}, std={STD_RATIO}) — Entfernte Punkte rot',
    scene=dict(
        xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
        aspectmode='data',
    ),
    height=600,
)
fig.show()

In [ ]:
# --- Mean kNN-Abstand: Entscheidungsgrundlage des Filters ---
pcd_tree = o3d.geometry.KDTreeFlann(pcd_raw)
mean_knn = np.zeros(len(pts_all))
for i, pt in enumerate(pts_all):
    _, idx, dists = pcd_tree.search_knn_vector_3d(pt, NB_NEIGHBORS + 1)
    mean_knn[i] = np.mean(np.sqrt(dists[1:]))

mean_knn_inlier = mean_knn[list(inlier_set)]
mean_knn_outlier = mean_knn[outlier_idx]

global_mean_k = mean_knn.mean()
global_std_k = mean_knn.std()
threshold_k = global_mean_k + STD_RATIO * global_std_k

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=mean_knn_inlier, nbinsx=100, name='Beibehalten',
    marker_color='#78909C', opacity=0.7,
))
fig.add_trace(go.Histogram(
    x=mean_knn_outlier, nbinsx=30, name='Entfernt',
    marker_color='#E53935', opacity=0.9,
))
fig.add_vline(
    x=threshold_k, line_dash='dash', line_color='#E53935', line_width=2,
    annotation_text=f'μ+{STD_RATIO}σ = {threshold_k:.3f}mm',
)
fig.update_layout(
    title=f'Mean {NB_NEIGHBORS}-NN Abstand — Entscheidungsgrundlage Statistical Outlier Filter',
    xaxis_title=f'Mittlerer Abstand zu {NB_NEIGHBORS} Nachbarn (mm)',
    yaxis_title='Anzahl',
    barmode='overlay', height=400,
)
fig.show()

print(f'Mean {NB_NEIGHBORS}-NN: μ={global_mean_k:.4f}, σ={global_std_k:.4f} mm')
print(f'Threshold: {threshold_k:.4f} mm')

In [ ]:
# --- Z-Verteilung der Outlier: Oberfläche vs. Spalt ---
n_surface = np.sum(pts_outlier[:, 2] >= -0.5)
n_gap = np.sum(pts_outlier[:, 2] < -0.5)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Seitenansicht (XZ)', 'Z-Verteilung Outlier'])

fig.add_trace(go.Scatter(
    x=pts_inlier[:, 0], y=pts_inlier[:, 2],
    mode='markers', marker=dict(size=1, color='#E0E0E0', opacity=0.3),
    name='Beibehalten', showlegend=False,
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=pts_outlier[:, 0], y=pts_outlier[:, 2],
    mode='markers', marker=dict(size=5, color='#E53935', opacity=0.9),
    name='Entfernt',
), row=1, col=1)

fig.add_trace(go.Histogram(
    y=pts_outlier[:, 2], nbinsy=40,
    marker_color='#E53935', opacity=0.8,
    name='Outlier Z', showlegend=False,
), row=1, col=2)
fig.add_hline(y=-0.5, line_dash='dot', line_color='gray', row=1, col=2,
              annotation_text=f'Oberfläche: {n_surface} | Spalt: {n_gap}')

fig.update_xaxes(title_text='X (mm)', row=1, col=1)
fig.update_yaxes(title_text='Z (mm)', row=1, col=1)
fig.update_xaxes(title_text='Anzahl', row=1, col=2)
fig.update_yaxes(title_text='Z (mm)', row=1, col=2)
fig.update_layout(height=500, title='Wo liegen die entfernten Punkte?')
fig.show()

### 5.2 – Radius Outlier Filter

Prüft für jeden Punkt, wie viele Nachbarn innerhalb eines festen Radius `search_radius` liegen.
Punkte mit weniger als `min_nb_points` Nachbarn im Radius werden entfernt.

→ Erwischt **einzelne isolierte Punkte** (z.B. Spritzer), die der statistische Filter ggf. nicht triggert.  
→ Bei KMM-Daten mit feinem Scan-Raster konservativ einstellen (`min_nb_points` niedrig halten).

In [ ]:
SEARCH_RADIUS = 1.5  # mm
MIN_NB_POINTS = 5

pcd_radius, inlier_idx_r = pcd_stat.remove_radius_outlier(
    nb_points=MIN_NB_POINTS, radius=SEARCH_RADIUS
)

all_idx_r = set(range(len(pcd_stat.points)))
inlier_set_r = set(inlier_idx_r)
outlier_idx_r = sorted(all_idx_r - inlier_set_r)

pts_stat = np.asarray(pcd_stat.points)
pts_inlier_r = pts_stat[list(inlier_set_r)]
pts_outlier_r = pts_stat[outlier_idx_r]

print(f'Vorher:   {len(pts_stat):,} Punkte')
print(f'Nachher:  {len(pts_inlier_r):,} Punkte')
print(f'Entfernt: {len(pts_outlier_r):,} ({len(pts_outlier_r)/len(pts_stat)*100:.1f}%)')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=pts_inlier_r[:, 0], y=pts_inlier_r[:, 1], z=pts_inlier_r[:, 2],
    mode='markers',
    marker=dict(size=1, color='#B0BEC5', opacity=0.3),
    name=f'Beibehalten ({len(pts_inlier_r):,})',
))
fig.add_trace(go.Scatter3d(
    x=pts_outlier_r[:, 0], y=pts_outlier_r[:, 1], z=pts_outlier_r[:, 2],
    mode='markers',
    marker=dict(size=4, color='#FF9800', opacity=0.9),
    name=f'Entfernt ({len(pts_outlier_r):,})',
))
fig.update_layout(
    title=f'Radius Outlier Filter (radius={SEARCH_RADIUS}mm, min_pts={MIN_NB_POINTS}) — Entfernte Punkte orange',
    scene=dict(
        xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
        aspectmode='data',
    ),
    height=600,
)
fig.show()

In [ ]:
n_surface_r = np.sum(pts_outlier_r[:, 2] >= -0.5)
n_gap_r = np.sum(pts_outlier_r[:, 2] < -0.5)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Seitenansicht (XZ)', 'Z-Verteilung Outlier'])

fig.add_trace(go.Scatter(
    x=pts_inlier_r[:, 0], y=pts_inlier_r[:, 2],
    mode='markers', marker=dict(size=1, color='#E0E0E0', opacity=0.3),
    name='Beibehalten', showlegend=False,
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=pts_outlier_r[:, 0], y=pts_outlier_r[:, 2],
    mode='markers', marker=dict(size=5, color='#FF9800', opacity=0.9),
    name='Entfernt',
), row=1, col=1)

fig.add_trace(go.Histogram(
    y=pts_outlier_r[:, 2], nbinsy=40,
    marker_color='#FF9800', opacity=0.8,
    name='Outlier Z', showlegend=False,
), row=1, col=2)
fig.add_hline(y=-0.5, line_dash='dot', line_color='gray', row=1, col=2,
              annotation_text=f'Oberfläche: {n_surface_r} | Spalt: {n_gap_r}')

fig.update_xaxes(title_text='X (mm)', row=1, col=1)
fig.update_yaxes(title_text='Z (mm)', row=1, col=1)
fig.update_xaxes(title_text='Anzahl', row=1, col=2)
fig.update_yaxes(title_text='Z (mm)', row=1, col=2)
fig.update_layout(height=500, title='Wo liegen die entfernten Punkte?')
fig.show()

### 5.3 – Voxel Grid Downsampling

Die Punktwolke wird in ein gleichmäßiges 3D-Raster mit Kantenlänge `voxel_size` unterteilt.
Pro Voxel bleibt ein Punkt (Schwerpunkt aller Punkte im Voxel).

→ Erzeugt **gleichmäßige Punktdichte** über die gesamte Wolke.  
→ `voxel_size` als Faustregel: 2–3× mittlerer Punktabstand. Bei 0.5mm bleiben die Details der V-Nut erhalten.

In [ ]:
VOXEL_SIZE = 0.5  # mm

pcd_voxel = pcd_radius.voxel_down_sample(voxel_size=VOXEL_SIZE)

pts_before_v = np.asarray(pcd_radius.points)
pts_after_v = np.asarray(pcd_voxel.points)

print(f'Vorher:   {len(pts_before_v):,} Punkte')
print(f'Nachher:  {len(pts_after_v):,} Punkte')
print(f'Entfernt: {len(pts_before_v) - len(pts_after_v):,} ({(1 - len(pts_after_v)/len(pts_before_v))*100:.1f}%)')

In [ ]:
# Entfernte Punkte per KDTree identifizieren
pcd_voxel_tree = o3d.geometry.KDTreeFlann(pcd_voxel)
kept_mask_v = np.zeros(len(pts_before_v), dtype=bool)
for i, pt in enumerate(pts_before_v):
    _, _, dist = pcd_voxel_tree.search_knn_vector_3d(pt, 1)
    if dist[0] < 1e-6:
        kept_mask_v[i] = True

pts_kept_v = pts_before_v[kept_mask_v]
pts_removed_v = pts_before_v[~kept_mask_v]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pts_kept_v[:, 0], y=pts_kept_v[:, 1],
    mode='markers', marker=dict(size=1.5, color='#B0BEC5', opacity=0.3),
    name=f'Beibehalten ({len(pts_kept_v):,})',
))
fig.add_trace(go.Scatter(
    x=pts_removed_v[:, 0], y=pts_removed_v[:, 1],
    mode='markers', marker=dict(size=1.5, color='#E53935', opacity=0.5),
    name=f'Entfernt ({len(pts_removed_v):,})',
))
fig.update_layout(
    title=f'Draufsicht (XY) — Voxel Downsampling ({VOXEL_SIZE}mm)',
    xaxis=dict(title='X (mm)', scaleanchor='y', scaleratio=1),
    yaxis=dict(title='Y (mm)'),
    height=600,
)
fig.show()

In [ ]:
# Querschnitt bei Y-Mitte: entfernte Punkte hervorgehoben
y_mid = np.median(pts_before_v[:, 1])
SLICE = 0.5  # mm

mask_slice = np.abs(pts_before_v[:, 1] - y_mid) < SLICE
slice_pts = pts_before_v[mask_slice]
slice_kept = kept_mask_v[mask_slice]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=slice_pts[slice_kept, 0], y=slice_pts[slice_kept, 2],
    mode='markers', marker=dict(size=3, color='steelblue', opacity=0.5),
    name=f'Beibehalten ({slice_kept.sum():,})',
))
fig.add_trace(go.Scatter(
    x=slice_pts[~slice_kept, 0], y=slice_pts[~slice_kept, 2],
    mode='markers', marker=dict(size=3, color='#E53935', opacity=0.7),
    name=f'Entfernt ({(~slice_kept).sum():,})',
))
fig.update_layout(
    title=f'Querschnitt bei Y≈{y_mid:.0f}mm (±{SLICE}mm) — Voxel Downsampling ({VOXEL_SIZE}mm)',
    xaxis_title='X (mm)', yaxis_title='Z (mm)',
    height=450,
)
fig.show()

In [ ]:
n_surface_v = np.sum(pts_removed_v[:, 2] >= -0.5)
n_gap_v = np.sum(pts_removed_v[:, 2] < -0.5)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Seitenansicht (XZ)', 'Z-Verteilung Entfernte'])

fig.add_trace(go.Scatter(
    x=pts_kept_v[:, 0], y=pts_kept_v[:, 2],
    mode='markers', marker=dict(size=1, color='#E0E0E0', opacity=0.3),
    name='Beibehalten', showlegend=False,
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=pts_removed_v[:, 0], y=pts_removed_v[:, 2],
    mode='markers', marker=dict(size=3, color='#E53935', opacity=0.8),
    name='Entfernt',
), row=1, col=1)

fig.add_trace(go.Histogram(
    y=pts_removed_v[:, 2], nbinsy=40,
    marker_color='#E53935', opacity=0.8,
    name='Entfernt Z', showlegend=False,
), row=1, col=2)
fig.add_hline(y=-0.5, line_dash='dot', line_color='gray', row=1, col=2,
              annotation_text=f'Oberfläche: {n_surface_v} | Spalt: {n_gap_v}')

fig.update_xaxes(title_text='X (mm)', row=1, col=1)
fig.update_yaxes(title_text='Z (mm)', row=1, col=1)
fig.update_xaxes(title_text='Anzahl', row=1, col=2)
fig.update_yaxes(title_text='Z (mm)', row=1, col=2)
fig.update_layout(height=500, title='Wo liegen die entfernten Punkte?')
fig.show()

### 5.4 – Normalenschätzung

Für jeden Punkt wird eine lokale Oberflächennormale geschätzt, basierend auf den Nachbarn im `radius`.
Die Normalen werden anschließend konsistent orientiert (`orient_mode='consistent'`).

→ **Voraussetzung für RANSAC-Segmentierung** (Ebenen erkennen anhand Normalenrichtung).  
→ Qualität hängt von `radius` relativ zur lokalen Geometrie ab: zu klein = verrauscht, zu groß = glättet Details weg.

In [ ]:
NORMAL_RADIUS = 2.0  # mm

pcd_normals = o3d.geometry.PointCloud(pcd_voxel)  # Kopie
pcd_normals.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamRadius(radius=NORMAL_RADIUS)
)
pcd_normals.orient_normals_consistent_tangent_plane(k=15)

print(f'Normalen berechnet: {pcd_normals.has_normals()}')
print(f'Punkte: {len(pcd_normals.points):,}')

In [ ]:
pts_n = np.asarray(pcd_normals.points)
normals_arr = np.asarray(pcd_normals.normals)

nz = normals_arr[:, 2]

fig = go.Figure(go.Scatter3d(
    x=pts_n[:, 0], y=pts_n[:, 1], z=pts_n[:, 2],
    mode='markers',
    marker=dict(size=1.5, color=nz, colorscale='RdBu', cmin=-1, cmax=1,
                colorbar=dict(title='Normal Z')),
    name=f'{len(pts_n):,} Punkte',
))
fig.update_layout(
    title=f'Geschätzte Normalen (Z-Komponente), radius={NORMAL_RADIUS}mm',
    scene=dict(
        xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
        aspectmode='data',
    ),
    height=600,
)
fig.show()

In [ ]:
# Histogramm der Normalenkomponenten
fig = make_subplots(rows=1, cols=3, subplot_titles=['Normal X', 'Normal Y', 'Normal Z'])

for i, (comp, label) in enumerate(zip(normals_arr.T, ['X', 'Y', 'Z'])):
    fig.add_trace(go.Histogram(
        x=comp, nbinsx=100, marker_color='steelblue', opacity=0.7,
        showlegend=False,
    ), row=1, col=i+1)

fig.update_layout(
    title='Verteilung der Normalenkomponenten — Peaks = dominante Flächenorientierungen',
    height=350,
)
fig.show()

print(f'Normal Z: mean={nz.mean():.3f}, median={np.median(nz):.3f}')
print(f'Anteil |nz| > 0.9 (≈flach): {(np.abs(nz) > 0.9).sum()} von {len(nz)} ({(np.abs(nz) > 0.9).mean()*100:.1f}%)')

---
## 6 – Geometrische Plausibilitätsprüfung (RANSAC)

Die dichtebasierten Filter (5.1–5.2) können Punkte nicht entfernen, die zwar räumlich nahe an der echten Geometrie
liegen, aber geometrisch "hinter" der V-Naht-Oberfläche liegen (z.B. Durchschuss des Messstrahls durch den Spalt).

**Ansatz:** Zwei Ebenen per RANSAC an die Spaltflanken fitten und Punkte verwerfen, die hinter beiden Ebenen liegen.

- Erwarteter Flankenwinkel: ~30° zur Vertikalen (60° V-Naht)
- Koordinatensystem: X = Stirnseite (Querschnitt), Y = Längsseite (Nahtrichtung), Z = Tiefe

In [ ]:
# --- Spaltbereich isolieren ---
pts_vox = np.asarray(pcd_voxel.points)

Z_THRESHOLD = -0.3  # mm — ab hier beginnt der Spalt
gap_mask = pts_vox[:, 2] < Z_THRESHOLD
pts_gap = pts_vox[gap_mask]

# Spaltmitte schätzen (X-Schwerpunkt der tiefsten Punkte)
deep_mask = pts_gap[:, 2] < np.percentile(pts_gap[:, 2], 10)
x_center = np.mean(pts_gap[deep_mask, 0])

left_mask = pts_gap[:, 0] < x_center
right_mask = ~left_mask

print(f'Punkte im Spaltbereich (Z < {Z_THRESHOLD}mm): {len(pts_gap):,}')
print(f'  Linke Flanke:  {left_mask.sum():,}')
print(f'  Rechte Flanke: {right_mask.sum():,}')
print(f'  Spaltmitte X ≈ {x_center:.1f} mm')

In [ ]:
# --- RANSAC Ebenenfit pro Flanke ---
DISTANCE_THRESHOLD = 0.15  # mm — max Abstand Punkt→Ebene für Inlier
RANSAC_N = 3
NUM_ITERATIONS = 1000

def fit_plane(points, distance_threshold=DISTANCE_THRESHOLD):
    """Fittet eine Ebene per RANSAC. Gibt (a,b,c,d), inlier_idx zurück."""
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    plane_model, inliers = pcd.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=RANSAC_N,
        num_iterations=NUM_ITERATIONS,
    )
    return plane_model, inliers

plane_left, inliers_left = fit_plane(pts_gap[left_mask])
plane_right, inliers_right = fit_plane(pts_gap[right_mask])

def plane_angle_to_z(plane):
    """Winkel der Ebenennormale zur Z-Achse in Grad."""
    a, b, c, d = plane
    normal = np.array([a, b, c])
    normal /= np.linalg.norm(normal)
    angle_to_z = np.degrees(np.arccos(np.abs(normal[2])))
    return angle_to_z, normal

angle_left, normal_left = plane_angle_to_z(plane_left)
angle_right, normal_right = plane_angle_to_z(plane_right)

print(f'Linke Flanke:  n=[{normal_left[0]:+.3f}, {normal_left[1]:+.3f}, {normal_left[2]:+.3f}]  '
      f'Winkel zur Z-Achse: {angle_left:.1f}° (erwartet: ~30°)')
print(f'Rechte Flanke: n=[{normal_right[0]:+.3f}, {normal_right[1]:+.3f}, {normal_right[2]:+.3f}]  '
      f'Winkel zur Z-Achse: {angle_right:.1f}° (erwartet: ~30°)')
print(f'Öffnungswinkel: {angle_left + angle_right:.1f}° (erwartet: ~60°)')

In [ ]:
# --- Punkte hinter beiden Ebenen markieren ---
def signed_distance(points, plane):
    """Vorzeichenbehafteter Abstand Punkt → Ebene."""
    a, b, c, d = plane
    return (a * points[:, 0] + b * points[:, 1] + c * points[:, 2] + d) / np.sqrt(a**2 + b**2 + c**2)

dist_left = signed_distance(pts_vox, plane_left)
dist_right = signed_distance(pts_vox, plane_right)

# Orientierung prüfen: Spaltmitte sollte positiven Abstand zu beiden Ebenen haben
test_point = np.array([[x_center, np.mean(pts_vox[:, 1]), np.min(pts_vox[:, 2])]])
sign_left = np.sign(signed_distance(test_point, plane_left)[0])
sign_right = np.sign(signed_distance(test_point, plane_right)[0])

# Punkt liegt hinter BEIDEN Flanken = geometrisch implausibel
behind_left = (dist_left * sign_left) > DISTANCE_THRESHOLD
behind_right = (dist_right * sign_right) > DISTANCE_THRESHOLD

# Nur Punkte im Spaltbereich betrachten (Oberfläche nicht anfassen)
implausible = gap_mask & behind_left & behind_right

print(f'Implausible Punkte: {implausible.sum():,} von {len(pts_vox):,} '
      f'({implausible.sum()/len(pts_vox)*100:.1f}%)')
print(f'Davon im Spaltbereich: {implausible.sum():,} von {gap_mask.sum():,} '
      f'({implausible.sum()/gap_mask.sum()*100:.1f}%)')

In [ ]:
# --- Querschnitt bei Y-Mitte mit gefitteten Ebenen ---
y_mid = np.median(pts_vox[:, 1])
SLICE = 1.0  # mm

slice_mask = np.abs(pts_vox[:, 1] - y_mid) < SLICE
sl = pts_vox[slice_mask]
sl_implausible = implausible[slice_mask]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=sl[~sl_implausible, 0], y=sl[~sl_implausible, 2],
    mode='markers', marker=dict(size=3, color='steelblue', opacity=0.5),
    name=f'Plausibel ({(~sl_implausible).sum():,})',
))
fig.add_trace(go.Scatter(
    x=sl[sl_implausible, 0], y=sl[sl_implausible, 2],
    mode='markers', marker=dict(size=4, color='#E53935', opacity=0.8),
    name=f'Implausibel ({sl_implausible.sum():,})',
))

# Ebenen als Linien im Querschnitt einzeichnen
x_range = np.linspace(sl[:, 0].min(), sl[:, 0].max(), 100)
a, b, c, d = plane_left
z_left = -(a * x_range + d) / c
a, b, c, d = plane_right
z_right = -(a * x_range + d) / c

fig.add_trace(go.Scatter(
    x=x_range, y=z_left, mode='lines',
    line=dict(color='#FF9800', width=2, dash='dash'),
    name=f'Flanke links ({angle_left:.0f}°)',
))
fig.add_trace(go.Scatter(
    x=x_range, y=z_right, mode='lines',
    line=dict(color='#4CAF50', width=2, dash='dash'),
    name=f'Flanke rechts ({angle_right:.0f}°)',
))

fig.update_layout(
    title=f'RANSAC Ebenenfit — Querschnitt bei Y≈{y_mid:.0f}mm (±{SLICE}mm)',
    xaxis_title='X (mm)', yaxis_title='Z (mm)',
    height=500,
)
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=pts_vox[~implausible, 0], y=pts_vox[~implausible, 1], z=pts_vox[~implausible, 2],
    mode='markers',
    marker=dict(size=1, color='#B0BEC5', opacity=0.3),
    name=f'Plausibel ({(~implausible).sum():,})',
))
fig.add_trace(go.Scatter3d(
    x=pts_vox[implausible, 0], y=pts_vox[implausible, 1], z=pts_vox[implausible, 2],
    mode='markers',
    marker=dict(size=3, color='#E53935', opacity=0.9),
    name=f'Implausibel ({implausible.sum():,})',
))
fig.update_layout(
    title=f'Geometrische Plausibilitätsprüfung — {implausible.sum():,} Punkte hinter beiden Flankenebenen',
    scene=dict(
        xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
        aspectmode='data',
    ),
    height=600,
)
fig.show()

In [ ]:
pts_plausible = pts_vox[~implausible]
pts_implausible = pts_vox[implausible]

n_surface_i = np.sum(pts_implausible[:, 2] >= -0.5)
n_gap_i = np.sum(pts_implausible[:, 2] < -0.5)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Seitenansicht (XZ)', 'Z-Verteilung Implausible'])

fig.add_trace(go.Scatter(
    x=pts_plausible[:, 0], y=pts_plausible[:, 2],
    mode='markers', marker=dict(size=1, color='#E0E0E0', opacity=0.3),
    name='Plausibel', showlegend=False,
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=pts_implausible[:, 0], y=pts_implausible[:, 2],
    mode='markers', marker=dict(size=3, color='#E53935', opacity=0.8),
    name='Implausibel',
), row=1, col=1)

fig.add_trace(go.Histogram(
    y=pts_implausible[:, 2], nbinsy=40,
    marker_color='#E53935', opacity=0.8,
    name='Implausibel Z', showlegend=False,
), row=1, col=2)
fig.add_hline(y=-0.5, line_dash='dot', line_color='gray', row=1, col=2,
              annotation_text=f'Oberfläche: {n_surface_i} | Spalt: {n_gap_i}')

fig.update_xaxes(title_text='X (mm)', row=1, col=1)
fig.update_yaxes(title_text='Z (mm)', row=1, col=1)
fig.update_xaxes(title_text='Anzahl', row=1, col=2)
fig.update_yaxes(title_text='Z (mm)', row=1, col=2)
fig.update_layout(height=500, title='Wo liegen die implausiblen Punkte?')
fig.show()

---
## 7 – Komplette Pipeline

Alle Steps zusammen als `PreprocessingPipeline`, analog zu `pipeline.yaml`.

In [ ]:
pipeline = PreprocessingPipeline(
    steps=[
        StatisticalOutlierFilter(nb_neighbors=20, std_ratio=2.0),
        RadiusOutlierFilter(search_radius=1.5, min_nb_points=5),
        VoxelGridDownsampler(voxel_size=0.5),
        NormalEstimator(radius=2.0, orient_mode='consistent'),
    ],
    source_type='real',
)
print(f'Pipeline: {pipeline}')

pcd_final, report = pipeline.process(pcd_raw)

print(f'\nReport:')
print(f'  Punkte rein:  {report.points_in:,}')
print(f'  Punkte raus:  {report.points_out:,}')
print(f'  Retention:    {report.total_retention_rate:.1%}')
print(f'  Gesamt-Zeit:  {report.total_duration_ms:.0f} ms')
print()
for s in report.steps:
    print(f'  {s.step_name:<35} {s.points_before:>8,} → {s.points_after:>8,}  '
          f'(−{s.points_removed:,}, {s.duration_ms:.1f}ms)')

In [ ]:
# Entfernte Punkte = in pcd_raw aber nicht in pcd_final
pts_raw_arr = np.asarray(pcd_raw.points)
pts_final_arr = np.asarray(pcd_final.points)

pcd_final_tree = o3d.geometry.KDTreeFlann(pcd_final)
removed_mask = np.zeros(len(pts_raw_arr), dtype=bool)
for i, pt in enumerate(pts_raw_arr):
    _, idx, dist = pcd_final_tree.search_knn_vector_3d(pt, 1)
    if dist[0] > 1e-6:
        removed_mask[i] = True

pts_kept = pts_raw_arr[~removed_mask]
pts_removed = pts_raw_arr[removed_mask]

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=pts_kept[:, 0], y=pts_kept[:, 1], z=pts_kept[:, 2],
    mode='markers',
    marker=dict(size=1, color='#B0BEC5', opacity=0.3),
    name=f'Beibehalten ({len(pts_kept):,})',
))
fig.add_trace(go.Scatter3d(
    x=pts_removed[:, 0], y=pts_removed[:, 1], z=pts_removed[:, 2],
    mode='markers',
    marker=dict(size=3, color='#E53935', opacity=0.9),
    name=f'Entfernt ({len(pts_removed):,})',
))
fig.update_layout(
    title=f'Komplette Pipeline: {len(pts_removed):,} entfernte Punkte hervorgehoben',
    scene=dict(
        xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
        aspectmode='data',
    ),
    height=600,
)
fig.show()

In [ ]:
# Ergebnis nach Pipeline + RANSAC Plausibilitätsprüfung
pts_final_clean = pts_vox[~implausible]
pcd_clean = o3d.geometry.PointCloud()
pcd_clean.points = o3d.utility.Vector3dVector(pts_final_clean)

plot_single(pcd_clean, f'Ergebnis: {len(pcd_clean.points):,} Punkte nach Preprocessing + RANSAC')

In [ ]:
# --- Komplette Kette: Stat Filter → RANSAC → Stat Filter 2 → Radius Filter → Voxel → Normalen ---

steps_log = []

# 1. Statistical Outlier Filter
n_in = len(pcd_raw.points)
pcd_pipe, _ = pcd_raw.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
steps_log.append(('Statistical Outlier', n_in, len(pcd_pipe.points)))

# 2. RANSAC Plausibilitätsprüfung
pts_pipe = np.asarray(pcd_pipe.points)
gap_m = pts_pipe[:, 2] < Z_THRESHOLD
pts_g = pts_pipe[gap_m]
deep_m = pts_g[:, 2] < np.percentile(pts_g[:, 2], 10)
xc = np.mean(pts_g[deep_m, 0])

pl_l, _ = fit_plane(pts_g[pts_g[:, 0] < xc])
pl_r, _ = fit_plane(pts_g[pts_g[:, 0] >= xc])

dl = signed_distance(pts_pipe, pl_l)
dr = signed_distance(pts_pipe, pl_r)
tp = np.array([[xc, np.mean(pts_pipe[:, 1]), np.min(pts_pipe[:, 2])]])
sl = np.sign(signed_distance(tp, pl_l)[0])
sr = np.sign(signed_distance(tp, pl_r)[0])
impl = gap_m & ((dl * sl) > DISTANCE_THRESHOLD) & ((dr * sr) > DISTANCE_THRESHOLD)

n_before_ransac = len(pts_pipe)
pcd_pipe = o3d.geometry.PointCloud()
pcd_pipe.points = o3d.utility.Vector3dVector(pts_pipe[~impl])
steps_log.append(('RANSAC Plausibilität', n_before_ransac, len(pcd_pipe.points)))

# 3. Statistical Outlier Filter (zweiter Durchlauf)
n_before = len(pcd_pipe.points)
pcd_pipe, _ = pcd_pipe.remove_statistical_outlier(nb_neighbors=10, std_ratio=1.5)
steps_log.append(('Statistical Outlier 2', n_before, len(pcd_pipe.points)))

# 4. Radius Outlier Filter
n_before = len(pcd_pipe.points)
pcd_pipe, _ = pcd_pipe.remove_radius_outlier(nb_points=5, radius=1.5)
steps_log.append(('Radius Outlier', n_before, len(pcd_pipe.points)))

# 5. Voxel Downsampling
n_before = len(pcd_pipe.points)
pcd_pipe = pcd_pipe.voxel_down_sample(voxel_size=0.5)
steps_log.append(('Voxel Downsampling', n_before, len(pcd_pipe.points)))

# 6. Normalenschätzung
pcd_pipe.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamRadius(radius=2.0))
pcd_pipe.orient_normals_consistent_tangent_plane(k=15)

pcd_final = pcd_pipe

# --- Report ---
n_raw = len(pcd_raw.points)
rows = [{'Step': 'Roh (Input)', 'Punkte': n_raw, 'Entfernt': 0, 'Retention (vs. Roh)': '100.0%'}]
for name, n_in, n_out in steps_log:
    rows.append({
        'Step': name,
        'Punkte': n_out,
        'Entfernt': n_in - n_out,
        'Retention (vs. Roh)': f'{n_out / n_raw:.1%}',
    })

df = pd.DataFrame(rows)
display(df)

fig = px.bar(
    df, x='Step', y='Punkte',
    title='Punktanzahl nach jedem Step (inkl. RANSAC)',
    color='Punkte', color_continuous_scale='Blues',
    text='Retention (vs. Roh)',
)
fig.update_traces(textposition='outside')
fig.update_layout(
    height=400,
    showlegend=False,
    yaxis=dict(range=[0, n_raw * 1.15]),
)
fig.show()

plot_single(pcd_final, f'Ergebnis: {len(pcd_final.points):,} Punkte nach kompletter Pipeline')

## 8 – Als WeldVolumeModel speichern

In [ ]:
model = WeldVolumeModel(
    model_id=SCAN_FILE.stem,
    source_type='real',
    source_file=SCAN_FILE,
    point_cloud=pcd_final,
    preprocessing_report=report,
    metadata={
        'scanner': 'Hexagon CMM (PC-DMIS)',
        'scan_directions': len(unique_normals),
        'raw_points': len(pcd_raw.points),
    },
)

save_path = model.save(Path('../data/processed/cmm_scans'))
print(f'Gespeichert: {save_path}')
print(f'Modell:      {model}')

---
## 9 – Nächste Schritte

- [ ] Spalte 7 im .xyz klären (Messunsicherheit? Intensität?)
- [ ] Preprocessing-Parameter an reale Daten tunen (bes. `voxel_size` relativ zur KMM-Auflösung)
- [ ] RANSAC-Parameter tunen (`Z_THRESHOLD`, `DISTANCE_THRESHOLD`) anhand weiterer Scans
- [ ] Vergleich: KMM-Scan vs. CAD-Ideal (AP2.2 Subtraktionsmethode)
- [ ] Mehrere Proben vergleichen (Batch-Exploration)